# Phase 2 — PoT & NoT Evaluation on Arithmetic Datasets

This notebook evaluates two complementary reasoning methods across two arithmetic datasets:

| Method | ID | Description |
|--------|----|-------------|
| **PoT** (Program-of-Thought) | Case 1 | LLM extracts JSON params → deterministic Python executor computes answer |
| **NoT** (Narrative-of-Thought) | Case 2 | Single-pass structured prompt: STRUCTURE → NARRATIVE → ANSWER |

| Dataset | Source | Format | N samples | Question types |
|---------|--------|--------|-----------|----------------|
| `arithmetic_saq` | `arithmetic_saq.csv` | CSV | 15 584 | Hour Adj, Month/Year Shift, Date Comp, Time Comp, TZ Conv, Week ID, Application |
| `test_adjusted` | `test_adjusted.json` | JSON | 1 850 | add_subtract, compare, duration, multi_op, schedule, timezone, trick |

**Design**: each experiment lives in **one independent cell** — re-running a cell only affects that experiment. The model loads once in Setup 4 and is reused across all 6 experiments.

**PoT pipeline**:
```
Question → [LLM: extract JSON params] → [Python executor] → Answer
                              ↓ (if executor returns None)
                       [LLM fallback answer extraction]
```

**NoT pipeline** (single-inference variant):
```
Question → [LLM: STRUCTURE + NARRATIVE + ANSWER in one call] → regex extract ANSWER:
```

## Setup (run sequentially once)

In [ ]:
# === SETUP 1 — install packages ===
!pip install -q -U transformers accelerate bitsandbytes scikit-learn pyyaml pandas

In [ ]:
# === SETUP 2 — mount Drive + clone repo ===
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_URL = 'https://github.com/<YOUR_USER>/Temporal_Reasoning.git'  # TODO: change
REPO_DIR = '/content/Temporal_Reasoning'
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# === SETUP 3 — paths ===
import os, sys

REPO_DIR       = '/content/Temporal_Reasoning'
DRIVE_OUT      = '/content/drive/MyDrive/Temporal_Reasoning/outputs'
ARITHMETIC_DIR = '/content/drive/MyDrive/Temporal_Reasoning/arithmetic'

os.makedirs(DRIVE_OUT, exist_ok=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

SAQ_PATH      = os.path.join(ARITHMETIC_DIR, 'arithmetic_saq.csv')
SAQ_SHOTS     = os.path.join(ARITHMETIC_DIR, 'arithmetic_shots_saq.csv')
TEST_ADJ_PATH = os.path.join(ARITHMETIC_DIR, 'test_adjusted.json')

print('DRIVE_OUT    :', DRIVE_OUT)
print('SAQ_PATH     :', SAQ_PATH)
print('TEST_ADJ_PATH:', TEST_ADJ_PATH)

In [ ]:
# === SETUP 4 — load model (GemmaChatLM, shared across all experiments) ===
import yaml
from pathlib import Path
from src.models.gemma import GemmaChatLM, GemmaConfig

# google/gemma-4-E4B-it in bfloat16 fits on a Colab T4 (~8 GB VRAM).
# Set load_in_4bit=True if you hit OOM on a smaller runtime.
MODEL = GemmaChatLM(GemmaConfig(
    model_name='google/gemma-4-E4B-it',
    dtype='bfloat16',
    load_in_4bit=False,
))
MODEL.load()
print('Model ready:', MODEL.config.model_name)

In [ ]:
# === SETUP 5 — dataset loaders ===
import json
import random
import pandas as pd

def load_arithmetic_saq(path, max_samples=None, seed=42):
    """Load arithmetic_saq CSV → list of sample dicts."""
    df = pd.read_csv(path)
    if max_samples and max_samples < len(df):
        df = df.sample(n=max_samples, random_state=seed).reset_index(drop=True)
    return [
        {
            'question': str(row['Question']),
            'gold':     str(row['Answer']).strip(),
            'category': str(row['Category']),
            'dataset':  'arithmetic_saq',
        }
        for _, row in df.iterrows()
    ]

def load_test_adjusted(path, max_samples=None, seed=42):
    """Load test_adjusted JSON → list of sample dicts.

    The 'label' field is a stringified Python dict, e.g.
        "{'answer': '352 BC'}" or '{"H": 2.0, "M": 13.0, "S": 30.0}'
    We keep it as-is and parse it during evaluation.
    """
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    if max_samples and max_samples < len(data):
        random.seed(seed)
        data = random.sample(data, max_samples)
    return [
        {
            'question': item['question'],
            'gold':     item['label'],
            'category': item['question_type'],
            'dataset':  'test_adjusted',
        }
        for item in data
    ]

def load_from_config(cfg: dict):
    name = cfg['dataset']
    kw = dict(max_samples=cfg.get('max_samples'), seed=cfg.get('seed', 42))
    if name == 'arithmetic_saq':
        return load_arithmetic_saq(cfg.get('dataset_path', SAQ_PATH), **kw)
    if name == 'test_adjusted':
        return load_test_adjusted(cfg.get('dataset_path', TEST_ADJ_PATH), **kw)
    raise ValueError(f'Unknown dataset: {name!r}')

# Quick sanity check
saq_preview = load_arithmetic_saq(SAQ_PATH, max_samples=3)
for s in saq_preview:
    print(f"[{s['category']}] Q: {s['question'][:70]}  A: {s['gold']!r}")
print()
adj_preview = load_test_adjusted(TEST_ADJ_PATH, max_samples=3)
for s in adj_preview:
    print(f"[{s['category']}] Q: {s['question'][:70]}  A: {s['gold']!r}")

In [ ]:
# === SETUP 6 — PoT pipeline ===
# Stage 1: LLM extracts JSON parameters (no computation).
# Stage 2: Python executor computes deterministically.
# Fallback: if executor returns None, LLM is asked for the direct answer.

import re
import json as _json
from datetime import datetime, timedelta
from src.models.base import ChatMessage

# ── Extraction system prompt ──────────────────────────────────────────────────
_POT_SYSTEM = (
    'You are a temporal parameter extractor. '
    'Given a question, output ONLY a single valid JSON object with the extracted parameters. '
    'Do NOT compute or guess the answer — only extract the values stated in the question.'
)

# Schema hints guide the LLM towards the right field names per category.
_SCHEMA_HINTS = {
    # ── arithmetic_saq categories ─────────────────────────────────────────────
    'Hour Adjustment (24h)':  '{"task":"hour_24h", "time1":"HH:MM", "op":"+"/"-", "time2":"HH:MM"}',
    'Hour Adjustment (12h)':  '{"task":"hour_12h", "time1":"HH:MM", "period":"AM"/"PM", "op":"+"/"-", "time2":"HH:MM"}',
    'Month Shift':            '{"task":"month_shift", "n":<int>, "direction":"before"/"after", "month":"<name>"}',
    'Year Shift':             '{"task":"year_shift", "n":<int>, "direction":"before"/"after", "year":<int>}',
    'Time Computation':       '{"task":"time_computation", "op":"add"/"subtract", "base_h":<int>, "base_m":<int>, "base_s":0, "delta_h":0, "delta_m":<int>, "delta_s":<int>}',
    'Date Computation':       '{"task":"date_computation", "base":"<Month YYYY or MM-DD-YYYY>", "delta_years":<int>, "delta_months":<int>, "delta_days":0}',
    'Time Zone Conversion':   '{"task":"tz_conversion", "datetime_str":"<str>", "source_tz":"<name>", "target_tz":"<name>"}',
    'Week Identification':    '{"task":"week_id", "day":<int>, "month":<int>, "year":<int>}',
    'Application':            '{"task":"application", "sub_type":"speed_dist"/"age"/"flight", "val1":<num>, "val2":<num>, "unit1":"<str>", "answer_unit":"<str>"}',
    # ── test_adjusted question_types ──────────────────────────────────────────
    'add_subtract':           '{"task":"add_subtract", "start":<int>, "start_era":"BC"/"AD", "delta":<int>, "op":"add"/"subtract"}',
    'compare':                '{"task":"compare", "question_asks":"<earliest/latest/in_range>", "entities":[{"name":"<str>", "datetime":"<str>"}]}',
    'duration':               '{"task":"duration", "p1_name":"<str>", "p1_birth":"<YYYY-Mon-DD>", "p2_name":"<str>", "p2_birth":"<YYYY-Mon-DD>", "condition_age_days":<int>, "condition_person":"p1"/"p2"}',
    'multi_op':               '{"task":"multi_op", "times":["H:MM:SS",...], "op":"average"/"sum"/"difference"}',
    'schedule':               '{"task":"schedule", "persons":[{"name":"<str>", "slots":[["HH:MM","HH:MM"],...]}], "meeting_minutes":<int>}',
    'timezone':               '{"task":"timezone_diff", "tz1_name":"<str>", "tz1_offset_h":<float>, "tz2_name":"<str>", "tz2_offset_h":<float>}',
    'trick':                  '{"task":"trick_date", "anchor_phrase":"<e.g. day after yesterday>", "anchor_date":"<YYYY-Mon-DD>", "delta_days":<int>}',
}

def _pot_extract(model, question: str, category: str) -> dict | None:
    """Ask LLM to extract parameters as JSON (Stage 1)."""
    hint = _SCHEMA_HINTS.get(category, '{"task":"unknown"}')
    user = f'Schema: {hint}\n\nQuestion: {question}\n\nJSON:'
    msgs = [
        ChatMessage(role='system', content=_POT_SYSTEM),
        ChatMessage(role='user',   content=user),
    ]
    raw = model.generate(msgs, max_new_tokens=300, temperature=0.0,
                         do_sample=False, enable_thinking=False)
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if not m:
        return None
    try:
        return _json.loads(m.group())
    except Exception:
        return None


# ── Python executor (Stage 2) ─────────────────────────────────────────────────
_MONTHS = ['January','February','March','April','May','June',
           'July','August','September','October','November','December']

# UTC offsets for timezone executor (using DST-aware summer offsets to match dataset)
_TZ_OFFSET_H = {
    'UTC': 0, 'GMT': 0,
    'US/Eastern': -5, 'EST': -5, 'EDT': -5,
    'US/Central': -6, 'CST': -6, 'CDT': -6,
    'US/Mountain': -6, 'MST': -7, 'MDT': -6,   # MDT=-6 matches dataset samples
    'US/Pacific': -7, 'PST': -8, 'PDT': -7,
    'US/Hawaii': -10, 'HST': -10,
    'Europe/London': 0, 'BST': 1,
    'Europe/Paris': 0, 'CET': 1, 'CEST': 2,    # CET+0 approximation for historical
    'Europe/Athens': 2, 'EET': 2, 'EEST': 3,
    'Asia/Shanghai': 8, 'CST_CHINA': 8,
    'Asia/Tokyo': 9, 'JST': 9,
    'Asia/Kolkata': 5.5, 'IST': 5.5,
    'Australia/Sydney': 10, 'AEST': 10,
}

def _execute_params(params: dict) -> str | None:
    """Deterministic Python executor for extracted params."""
    task = params.get('task', '')

    # ── Hour Adjustment 24h ──────────────────────────────────────────────────
    if task == 'hour_24h':
        try:
            h1, m1 = map(int, str(params['time1']).split(':'))
            h2, m2 = map(int, str(params['time2']).split(':'))
            op = params.get('op', '-')
            base, delta = h1 * 60 + m1, h2 * 60 + m2
            result = (base + delta) % 1440 if op == '+' else (base - delta) % 1440
            return f'{result // 60:02d}:{result % 60:02d}'
        except Exception:
            return None

    # ── Hour Adjustment 12h ──────────────────────────────────────────────────
    elif task == 'hour_12h':
        try:
            h, m = map(int, str(params['time1']).split(':'))
            period = str(params.get('period', 'PM')).upper()
            h2, m2 = map(int, str(params['time2']).split(':'))
            op = params.get('op', '-')
            if period == 'PM' and h != 12:
                h += 12
            elif period == 'AM' and h == 12:
                h = 0
            base, delta = h * 60 + m, h2 * 60 + m2
            result = (base + delta) % 1440 if op == '+' else (base - delta) % 1440
            rh, rm = result // 60, result % 60
            p_out = 'AM' if rh < 12 else 'PM'
            rh12 = rh % 12 or 12
            return f'{rh12}:{rm:02d} {p_out}'
        except Exception:
            return None

    # ── Month Shift ──────────────────────────────────────────────────────────
    elif task == 'month_shift':
        try:
            n = int(params['n'])
            direction = str(params.get('direction', 'after')).lower()
            month_name = str(params['month'])
            idx = next((i for i, mn in enumerate(_MONTHS)
                        if mn.lower() == month_name.lower()), None)
            if idx is None:
                return None
            result = (idx + n) % 12 if direction == 'after' else (idx - n) % 12
            return _MONTHS[result]
        except Exception:
            return None

    # ── Year Shift ────────────────────────────────────────────────────────────
    elif task == 'year_shift':
        try:
            n = int(params['n'])
            direction = str(params.get('direction', 'after')).lower()
            year = int(params['year'])
            return str(year + n if direction == 'after' else year - n)
        except Exception:
            return None

    # ── Time Computation ─────────────────────────────────────────────────────
    elif task == 'time_computation':
        try:
            op = str(params.get('op', 'subtract')).lower()
            bh = int(params.get('base_h', 0))
            bm = int(params.get('base_m', 0))
            bs = int(params.get('base_s', 0))
            dh = int(params.get('delta_h', 0))
            dm = int(params.get('delta_m', 0))
            ds = int(params.get('delta_s', 0))
            base_s  = bh * 3600 + bm * 60 + bs
            delta_s = dh * 3600 + dm * 60 + ds
            result  = base_s + delta_s if op == 'add' else base_s - delta_s
            if result < 0:
                return 'Invalid'
            rh, rem = divmod(result, 3600)
            rm, rs  = divmod(rem, 60)
            if rh > 0:
                return f'{rh} hours {rm} minutes {rs} seconds' if rs else f'{rh} hours {rm} minutes'
            return f'{rm} minutes {rs} seconds' if rs else f'{rm} minutes'
        except Exception:
            return None

    # ── Date Computation ─────────────────────────────────────────────────────
    elif task == 'date_computation':
        try:
            base_str = str(params['base'])
            dy = int(params.get('delta_years', 0))
            dm = int(params.get('delta_months', 0))
            dd = int(params.get('delta_days', 0))
            # Try parsing base date
            base_dt = None
            for fmt in ('%m-%d-%Y', '%B %Y', '%b %Y', '%B %d, %Y'):
                try:
                    base_dt = datetime.strptime(base_str, fmt)
                    break
                except ValueError:
                    continue
            if base_dt is None:
                return None
            if dd:
                base_dt += timedelta(days=dd)
            # Add years and months
            total_m = base_dt.month + dm + dy * 12
            year  = base_dt.year + (total_m - 1) // 12
            month = (total_m - 1) % 12 + 1
            result_dt = datetime(year, month, min(base_dt.day, 28))
            if dd and not (dy or dm):
                return result_dt.strftime('%B %d, %Y')
            return result_dt.strftime('%B %Y')
        except Exception:
            return None

    # ── Week Identification ───────────────────────────────────────────────────
    elif task == 'week_id':
        try:
            d = datetime(int(params['year']), int(params['month']), int(params['day']))
            week_num = d.isocalendar()[1]
            return f'Week {week_num}'
        except Exception:
            return None

    # ── Timezone Conversion (arithmetic_saq) ─────────────────────────────────
    elif task == 'tz_conversion':
        try:
            src = str(params.get('source_tz', ''))
            tgt = str(params.get('target_tz', ''))
            dt_str = str(params.get('datetime_str', ''))
            src_off = _TZ_OFFSET_H.get(src, _TZ_OFFSET_H.get(src.split('/')[-1], None))
            tgt_off = _TZ_OFFSET_H.get(tgt, _TZ_OFFSET_H.get(tgt.split('/')[-1], None))
            if src_off is None or tgt_off is None:
                return None
            # Parse hour from datetime_str, e.g. "4 AM on August 5, 1180"
            hm = re.search(r'(\d{1,2})\s*(AM|PM)', dt_str, re.I)
            date_m = re.search(
                r'(January|February|March|April|May|June|July|August|September|October|November|December)\s+(\d+)[,\s]+(\d+)',
                dt_str, re.I
            )
            if not hm:
                return None
            h24 = int(hm.group(1))
            per = hm.group(2).upper()
            if per == 'PM' and h24 != 12:
                h24 += 12
            elif per == 'AM' and h24 == 12:
                h24 = 0
            # Apply offset difference
            diff_h = int(tgt_off - src_off)
            new_h = (h24 + diff_h) % 24
            day_delta = (h24 + diff_h) // 24  # 0 or ±1
            # Format result
            p_out = 'AM' if new_h < 12 else 'PM'
            rh12 = new_h % 12 or 12
            date_part = ''
            if date_m:
                month_n = date_m.group(1)
                day_n   = int(date_m.group(2)) + day_delta
                year_n  = date_m.group(3)
                date_part = f' on {month_n} {day_n}, {year_n}'
            return f'{rh12} {p_out}{date_part}'
        except Exception:
            return None

    # ── Application (speed-distance-time) ────────────────────────────────────
    elif task == 'application':
        try:
            sub = str(params.get('sub_type', 'speed_dist'))
            v1 = float(params.get('val1', 1))
            v2 = float(params.get('val2', 1))
            ans_unit = str(params.get('answer_unit', 'minutes')).lower()
            if sub == 'speed_dist':
                hours = v2 / v1  # distance / speed
                if 'minute' in ans_unit:
                    r = round(hours * 60, 2)
                    return str(int(r) if r == int(r) else r)
                return str(round(hours, 2))
            return None  # other sub-types fall back to LLM
        except Exception:
            return None

    # ── test_adjusted: add_subtract ───────────────────────────────────────────
    elif task == 'add_subtract':
        try:
            start = int(params['start'])
            era   = str(params.get('start_era', 'AD')).upper()
            delta = int(params['delta'])
            op    = str(params.get('op', 'add')).lower()
            if era == 'BC':
                # BC years count backward: 360 BC + 8 years = 352 BC
                res = start - delta if op == 'add' else start + delta
                res_era = 'BC' if res > 0 else 'AD'
                return f'{abs(res)} {res_era}'
            else:
                res = start + delta if op == 'add' else start - delta
                res_era = 'AD' if res > 0 else 'BC'
                return f'{abs(res)} {res_era}'
        except Exception:
            return None

    # ── test_adjusted: timezone_diff ─────────────────────────────────────────
    elif task == 'timezone_diff':
        try:
            off1 = float(params.get('tz1_offset_h', 0))
            off2 = float(params.get('tz2_offset_h', 0))
            diff = abs(off1 - off2)
            h = int(diff)
            m = int((diff - h) * 60)
            return f'{h} hours {m} minutes'
        except Exception:
            return None

    # ── test_adjusted: trick_date ─────────────────────────────────────────────
    elif task == 'trick_date':
        try:
            anchor_str = str(params.get('anchor_date', ''))
            phrase     = str(params.get('anchor_phrase', '')).lower()
            delta      = int(params.get('delta_days', 0))
            # Parse anchor date
            anchor_dt = None
            for fmt in ('%Y-%b-%d', '%Y-%m-%d', '%Y-%B-%d'):
                try:
                    anchor_dt = datetime.strptime(anchor_str, fmt)
                    break
                except ValueError:
                    continue
            if anchor_dt is None:
                return None
            # Resolve reference phrase to actual "today"
            if 'day after yesterday' in phrase:
                today = anchor_dt  # anchor IS today
            elif 'yesterday' in phrase:
                today = anchor_dt + timedelta(days=1)
            elif 'tomorrow' in phrase:
                today = anchor_dt - timedelta(days=1)
            elif 'day before tomorrow' in phrase:
                today = anchor_dt  # anchor IS today
            else:
                today = anchor_dt
            result = today + timedelta(days=delta)
            return result.strftime('%Y-%m-%d')
        except Exception:
            return None

    return None  # unsupported task type


# ── LLM fallback (when executor returns None) ─────────────────────────────────
_POT_FALLBACK_SYSTEM = (
    'Solve the following time/date arithmetic question. '
    'Think briefly, then output ONLY the final answer on the last line, '
    'prefixed with "Answer:".'
)

def _pot_fallback(model, question: str) -> str | None:
    msgs = [
        ChatMessage(role='system', content=_POT_FALLBACK_SYSTEM),
        ChatMessage(role='user',   content=f'Question: {question}'),
    ]
    raw = model.generate(msgs, max_new_tokens=150, temperature=0.0,
                         do_sample=False, enable_thinking=False)
    m = re.search(r'Answer:\s*(.+?)(?:\n|$)', raw, re.IGNORECASE)
    return m.group(1).strip() if m else raw.strip().split('\n')[-1].strip()


def pot_predict(model, sample: dict) -> dict:
    """Full PoT pipeline for one sample."""
    question = sample['question']
    category = sample.get('category', '')

    params   = _pot_extract(model, question, category)
    executed = _execute_params(params) if params else None
    fallback_used = False

    if executed is None:
        executed = _pot_fallback(model, question)
        fallback_used = True

    return {
        'question':      question,
        'category':      category,
        'gold':          sample['gold'],
        'params':        params,
        'predicted':     executed,
        'fallback_used': fallback_used,
        'method':        'pot',
    }

print('PoT pipeline ready.')

In [ ]:
# === SETUP 7 — NoT (Narrative-of-Thought) pipeline ===
# Single-inference variant: three reasoning steps in ONE LLM call.
# Output format enforced via system prompt:
#   STRUCTURE:\n<facts>\n\nNARRATIVE:\n<reasoning>\n\nANSWER: <final answer>

import re
from src.models.base import ChatMessage

_NOT_SYSTEM = """You are a temporal reasoning assistant using Narrative-of-Thought (NoT).

Solve the question through three sequential steps:

STEP 1 — STRUCTURE:
List all temporal entities, dates, times, durations, and numeric values as explicit key-value facts. Include units.

STEP 2 — NARRATIVE:
Build a brief chronological narrative that traces all operations. Show each arithmetic step explicitly (e.g. 23:48 minus 1h31m = 22:17). For multi-entity questions, surface overlaps, orderings, and interval relationships.

STEP 3 — ANSWER:
State the final answer grounded in your narrative. Be concise and match the expected format.

You MUST use this exact format:
STRUCTURE:
<facts>

NARRATIVE:
<reasoning>

ANSWER: <exact final answer>"""

# Category-specific answer format reminders injected into the user message.
_NOT_FORMAT_HINTS = {
    # arithmetic_saq
    'Hour Adjustment (24h)':  'Answer format: HH:MM (24-hour, zero-padded)',
    'Hour Adjustment (12h)':  'Answer format: H:MM AM or H:MM PM',
    'Year Shift':             'Answer format: plain 4-digit year',
    'Month Shift':            'Answer format: full month name (e.g. July)',
    'Date Computation':       'Answer format: Month YYYY (e.g. October 1806)',
    'Week Identification':    'Answer format: Week N (e.g. Week 47)',
    'Time Zone Conversion':   'Answer format: H AM/PM on Month D, YYYY',
    'Time Computation':       'Answer format: N minutes M seconds  OR  N hours M minutes [M seconds]',
    'Application':            'Answer format: numeric value with no units (or HH:MM AM/PM for flight times)',
    # test_adjusted
    'add_subtract':           'Answer format: YYYY BC or YYYY AD (e.g. 352 BC)',
    'compare':                'Answer format: comma-separated list of names (unordered)',
    'duration':               'Answer format: integer number of days',
    'multi_op':               'Answer format: H hours M minutes S seconds  OR  H:M:S',
    'schedule':               'Answer format: integer count of valid meeting slots',
    'timezone':               'Answer format: N hours M minutes  OR  same_day/next_day HH:MM',
    'trick':                  'Answer format: YYYY-MM-DD',
}

def not_predict(model, sample: dict) -> dict:
    """Full NoT pipeline for one sample."""
    question = sample['question']
    category = sample.get('category', '')

    fmt_hint = _NOT_FORMAT_HINTS.get(category, '')
    user_content = f'Question: {question}'
    if fmt_hint:
        user_content += f'\n\n{fmt_hint}'

    msgs = [
        ChatMessage(role='system', content=_NOT_SYSTEM),
        ChatMessage(role='user',   content=user_content),
    ]
    raw = model.generate(msgs, max_new_tokens=600, temperature=0.0,
                         do_sample=False, enable_thinking=False)

    # Extract the ANSWER: line (last one wins if multiple)
    matches = re.findall(r'ANSWER:\s*(.+?)(?:\n|$)', raw, re.IGNORECASE)
    predicted = matches[-1].strip() if matches else None

    return {
        'question':   question,
        'category':   category,
        'gold':       sample['gold'],
        'raw_output': raw,
        'predicted':  predicted,
        'method':     'not',
    }

print('NoT pipeline ready.')

In [ ]:
# === SETUP 8 — evaluation helpers + run_exp() ===
import json, csv, time
from pathlib import Path

# ── Gold label parser ─────────────────────────────────────────────────────────
def _parse_gold(gold_raw):
    """Parse gold label into a canonical form for comparison.

    arithmetic_saq: plain string (e.g. '22:17', 'July', 'Week 47')
    test_adjusted : stringified dict like "{'answer': '352 BC'}" or
                    '{"H": 2.0, "M": 13.0, "S": 30.0}'
    Returns a string or sorted list-of-strings for unordered_list.
    """
    if not isinstance(gold_raw, str):
        return str(gold_raw).strip()
    s = gold_raw.strip()
    if s.startswith('{') or s.startswith("{"):
        try:
            g = eval(s)  # handles both JSON and Python dict literal syntax
        except Exception:
            return s
        if isinstance(g, dict):
            if 'answer' in g:
                return str(g['answer']).strip()
            if 'unordered_list' in g:
                return sorted(str(x).strip() for x in g['unordered_list'])
            if set(g.keys()) <= {'H', 'M', 'S'}:
                # multi_op: format as H:M:S
                return f"{int(g.get('H',0))}:{int(g.get('M',0))}:{int(g.get('S',0))}"
            if set(g.keys()) <= {'hours', 'minutes'}:
                return f"{g.get('hours',0)} hours {g.get('minutes',0)} minutes"
            if set(g.keys()) <= {'day', 'time'}:
                return f"{g.get('day','')} {g.get('time','')}".strip()
            return json.dumps(g, sort_keys=True)
    return s


def _norm(val) -> str:
    """Light normalization: strip, lowercase, collapse whitespace."""
    if val is None:
        return ''
    return ' '.join(str(val).strip().lower().split())


def _is_correct(predicted, gold_raw) -> bool:
    """Compare predicted string to parsed gold label."""
    gold = _parse_gold(gold_raw)
    if isinstance(gold, list):
        # Unordered list: parse predicted as comma-separated names
        try:
            pred_list = sorted(_norm(x) for x in predicted.split(',') if x.strip())
        except Exception:
            pred_list = []
        gold_norm = sorted(_norm(x) for x in gold)
        return pred_list == gold_norm
    return _norm(predicted) == _norm(gold)


# ── run_exp() ─────────────────────────────────────────────────────────────────
def run_exp(cfg_path, predict_fn, *,
            verbose=True, verbose_first_n=5, verbose_every=10,
            running_score_every=25, output_dir=None):
    """
    Run one experiment end-to-end.

    predict_fn(model, sample) -> dict with key 'predicted' (str | None)
    Returns metrics dict.
    """
    with open(cfg_path, encoding='utf-8') as f:
        cfg = yaml.safe_load(f)

    method  = cfg.get('method', 'unknown')
    dataset = cfg.get('dataset', 'unknown')
    samples = load_from_config(cfg)
    n       = len(samples)

    out_dir = Path(output_dir or cfg.get('output_dir', 'outputs')) / method / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_file = out_dir / 'predictions.jsonl'

    print(f'[run_exp] experiment={cfg.get("experiment_name")} '
          f'method={method} dataset={dataset} n={n}')

    correct_n  = 0
    parse_fail = 0
    records    = []

    with open(pred_file, 'w', encoding='utf-8') as pf:
        for i, s in enumerate(samples):
            t0  = time.time()
            res = predict_fn(MODEL, s)
            elapsed = time.time() - t0

            pred    = res.get('predicted')
            correct = _is_correct(pred, s['gold'])
            if pred is None:
                parse_fail += 1
            if correct:
                correct_n += 1

            gold_display = _parse_gold(s['gold'])
            rec = {
                'idx':        i,
                'question':   s['question'],
                'category':   s.get('category', ''),
                'gold_raw':   s['gold'],
                'gold_norm':  str(gold_display) if not isinstance(gold_display, list)
                              else str(sorted(gold_display)),
                'predicted':  pred,
                'correct':    correct,
                'elapsed_sec': round(elapsed, 3),
            }
            # Include method-specific extras (params, raw_output, fallback_used)
            for k in ('params', 'raw_output', 'fallback_used'):
                if k in res:
                    rec[k] = res[k]

            records.append(rec)
            pf.write(json.dumps(rec, ensure_ascii=False) + '\n')
            pf.flush()

            # ── Verbose logging ───────────────────────────────────────────────
            if verbose:
                in_first = i < verbose_first_n
                periodic = verbose_every > 0 and (i + 1) % verbose_every == 0
                if in_first or periodic:
                    mark = '\u2713' if correct else '\u2717'
                    print(f'  [{i+1}/{n}] {mark} '
                          f'gold={_norm(gold_display)!r} pred={_norm(pred)!r} '
                          f'({elapsed:.2f}s)')
                    if in_first:
                        print(f'    Q: {s["question"][:110]}')
                        if method == 'pot' and rec.get('params'):
                            print(f'    params: {rec["params"]}')
                        if method == 'not' and rec.get('raw_output'):
                            print(f'    raw[:180]: {rec["raw_output"][:180]}')

            if running_score_every > 0 and (i + 1) % running_score_every == 0:
                acc = correct_n / (i + 1)
                print(f'  [{i+1}/{n}] running acc={acc:.3f} '
                      f'({correct_n}/{i+1}) parse_fail={parse_fail}')

    # ── Final metrics ─────────────────────────────────────────────────────────
    acc = correct_n / n if n else 0.0
    # Per-category breakdown
    from collections import defaultdict
    cat_stats = defaultdict(lambda: [0, 0])  # [correct, total]
    for r in records:
        cat = r.get('category', 'unknown')
        cat_stats[cat][1] += 1
        if r['correct']:
            cat_stats[cat][0] += 1
    cat_acc = {k: round(v[0]/v[1], 3) for k, v in cat_stats.items() if v[1]}

    metrics = {
        'accuracy':   round(acc, 4),
        'correct':    correct_n,
        'total':      n,
        'parse_fail': parse_fail,
        'per_category': cat_acc,
    }
    metric_file = out_dir / 'metrics.json'
    metric_file.write_text(
        json.dumps({'experiment': cfg.get('experiment_name'), 'method': method,
                    'dataset': dataset, 'metrics': metrics}, indent=2, ensure_ascii=False),
        encoding='utf-8'
    )

    # Append to summary.csv
    summary_path = Path(output_dir or cfg.get('output_dir', 'outputs')) / 'summary.csv'
    header_needed = not summary_path.exists()
    row = {
        'experiment': cfg.get('experiment_name'), 'method': method, 'dataset': dataset,
        'accuracy': metrics['accuracy'], 'correct': correct_n,
        'total': n, 'parse_fail': parse_fail,
    }
    with open(summary_path, 'a', encoding='utf-8', newline='') as sf:
        w = csv.DictWriter(sf, fieldnames=list(row.keys()))
        if header_needed:
            w.writeheader()
        w.writerow(row)

    print(f'\n── Results ─────────────────────────────────────────────')
    print(f'  accuracy     : {acc:.4f}  ({correct_n}/{n})')
    print(f'  parse_fail   : {parse_fail}')
    print(f'  per_category : {cat_acc}')
    print(f'  predictions  : {pred_file}')
    print(f'  metrics      : {metric_file}')
    print(f'  summary      : {summary_path}')
    print(f'─────────────────────────────────────────────────────────\n')
    return metrics

print('Evaluation helpers + run_exp() ready.')

In [ ]:
# === SETUP 9 — Hybrid router (PoT ↔ NoT per-category) ===
# Reads the routing table from a YAML config and dispatches each sample
# to either pot_predict or not_predict based on its category.
# Records which branch was taken, enabling per-route accuracy analysis.

import yaml

# ── Route resolver ────────────────────────────────────────────────────────────
def _load_routing(cfg: dict):
    """Return (routing_dict, default_route) from config."""
    return cfg.get("routing", {}), cfg.get("default_route", "not")

_ROUTE_LABEL = {"pot": "PoT", "not": "NoT"}


def hybrid_predict(model, sample: dict, routing: dict, default_route: str) -> dict:
    """Route sample to PoT or NoT based on category; return unified result dict."""
    category = sample.get("category", "")
    route = routing.get(category, default_route)

    if route == "pot":
        result = pot_predict(model, sample)
    else:
        result = not_predict(model, sample)

    result["routed_to"] = route
    result["method"]    = "hybrid"
    return result


def run_hybrid_exp(cfg_path, *, verbose=True, verbose_first_n=5,
                   verbose_every=10, running_score_every=25, output_dir=None):
    """run_exp() variant for hybrid: loads routing table and dispatches per sample."""
    import json, csv, time
    from pathlib import Path
    from collections import defaultdict

    with open(cfg_path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    routing, default_route = _load_routing(cfg)
    method  = "hybrid"
    dataset = cfg.get("dataset", "unknown")
    samples = load_from_config(cfg)
    n       = len(samples)

    out_dir = Path(output_dir or cfg.get("output_dir", "outputs")) / method / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    pred_file = out_dir / "predictions.jsonl"

    print(f"[run_hybrid_exp] experiment={cfg.get('experiment_name')} n={n}")
    print(f"  routing : {routing}")
    print(f"  default : {default_route}")

    correct_n = parse_fail = 0
    records   = []
    route_stats = defaultdict(lambda: [0, 0])   # route -> [correct, total]

    with open(pred_file, "w", encoding="utf-8") as pf:
        for i, s in enumerate(samples):
            t0  = time.time()
            res = hybrid_predict(MODEL, s, routing, default_route)
            elapsed = time.time() - t0

            pred    = res.get("predicted")
            route   = res.get("routed_to", default_route)
            correct = _is_correct(pred, s["gold"])
            if pred is None:
                parse_fail += 1
            if correct:
                correct_n += 1
            route_stats[route][1] += 1
            if correct:
                route_stats[route][0] += 1

            gold_display = _parse_gold(s["gold"])
            rec = {
                "idx":         i,
                "question":    s["question"],
                "category":    s.get("category", ""),
                "routed_to":   route,
                "gold_raw":    s["gold"],
                "gold_norm":   str(gold_display) if not isinstance(gold_display, list)
                               else str(sorted(gold_display)),
                "predicted":   pred,
                "correct":     correct,
                "elapsed_sec": round(elapsed, 3),
            }
            for k in ("params", "raw_output", "fallback_used"):
                if k in res:
                    rec[k] = res[k]

            records.append(rec)
            pf.write(json.dumps(rec, ensure_ascii=False) + "\n")
            pf.flush()

            # ── Verbose logging ───────────────────────────────────────────────
            if verbose:
                in_first = i < verbose_first_n
                periodic = verbose_every > 0 and (i + 1) % verbose_every == 0
                if in_first or periodic:
                    mark  = "\u2713" if correct else "\u2717"
                    badge = f"[{_ROUTE_LABEL.get(route, route.upper())}]"
                    print(f"  [{i+1}/{n}] {mark} {badge} "
                          f"cat={s['category']!r} "
                          f"gold={_norm(gold_display)!r} pred={_norm(pred)!r} "
                          f"({elapsed:.2f}s)")
                    if in_first:
                        print(f"    Q: {s['question'][:110]}")

            if running_score_every > 0 and (i + 1) % running_score_every == 0:
                acc = correct_n / (i + 1)
                rs  = {k: f"{v[0]}/{v[1]}" for k, v in route_stats.items()}
                print(f"  [{i+1}/{n}] running acc={acc:.3f} "
                      f"({correct_n}/{i+1}) per_route={rs} parse_fail={parse_fail}")

    # ── Final metrics ─────────────────────────────────────────────────────────
    acc = correct_n / n if n else 0.0

    per_route_acc = {
        k: {"acc": round(v[0]/v[1], 3), "correct": v[0], "total": v[1]}
        for k, v in route_stats.items() if v[1]
    }

    cat_stats = defaultdict(lambda: [0, 0])
    for r in records:
        cat = r.get("category", "unknown")
        cat_stats[cat][1] += 1
        if r["correct"]:
            cat_stats[cat][0] += 1
    cat_acc = {k: round(v[0]/v[1], 3) for k, v in cat_stats.items() if v[1]}

    metrics = {
        "accuracy":     round(acc, 4),
        "correct":      correct_n,
        "total":        n,
        "parse_fail":   parse_fail,
        "per_route":    per_route_acc,
        "per_category": cat_acc,
    }
    metric_file = out_dir / "metrics.json"
    metric_file.write_text(
        json.dumps({"experiment": cfg.get("experiment_name"), "method": method,
                    "dataset": dataset, "routing": routing,
                    "metrics": metrics}, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    summary_path = Path(output_dir or cfg.get("output_dir", "outputs")) / "summary.csv"
    header_needed = not summary_path.exists()
    row = {
        "experiment": cfg.get("experiment_name"), "method": method, "dataset": dataset,
        "accuracy": metrics["accuracy"], "correct": correct_n,
        "total": n, "parse_fail": parse_fail,
    }
    with open(summary_path, "a", encoding="utf-8", newline="") as sf:
        w = csv.DictWriter(sf, fieldnames=list(row.keys()))
        if header_needed:
            w.writeheader()
        w.writerow(row)

    print(f"\n── Results ──────────────────────────────────────────────")
    print(f"  accuracy    : {acc:.4f}  ({correct_n}/{n})")
    print(f"  per_route   : { {k: v['acc'] for k, v in per_route_acc.items()} }")
    print(f"  per_category: {cat_acc}")
    print(f"  parse_fail  : {parse_fail}")
    print(f"  predictions : {pred_file}")
    print(f"  metrics     : {metric_file}")
    print(f"─────────────────────────────────────────────────────────\n")
    return metrics

print("Hybrid router ready.")


## 6 Experiments — each cell is independent

Re-running any single cell only re-runs that experiment. Results are saved under `DRIVE_OUT/<method>/<dataset>/`.

### PoT — Program-of-Thought (Case 1)

Stage 1: LLM extracts structured JSON parameters from the question (no computation).  
Stage 2: Deterministic Python executor computes the answer.  
Fallback: If the executor returns `None`, the LLM is asked for a direct answer.

In [ ]:
# === EXP 1/4 — PoT × arithmetic_saq ===
# Dataset: arithmetic_saq.csv (Hour Adj, Month/Year Shift, Date Comp, Time Comp, TZ Conv, Week ID, Application)
# Metric : exact string match (accuracy)
m = run_exp(
    'configs/pot_arithmetic_saq.yaml',
    pot_predict,
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)

In [ ]:
# === EXP 2/4 — PoT × test_adjusted ===
# Dataset: test_adjusted.json (add_subtract, compare, duration, multi_op, schedule, timezone, trick)
# Metric : exact match with type-aware gold parsing (unordered list, H/M/S dict, etc.)
m = run_exp(
    'configs/pot_test_adjusted.yaml',
    pot_predict,
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)

### NoT — Narrative-of-Thought (Case 2)

Single-inference variant: LLM reasons through STRUCTURE → NARRATIVE → ANSWER in one call.  
For arithmetic_saq (pure computation), NoT acts as a structured CoT.  
For test_adjusted (multi-entity relational), NoT builds a chronological narrative before answering.

In [ ]:
# === EXP 3/4 — NoT × arithmetic_saq ===
m = run_exp(
    'configs/not_arithmetic_saq.yaml',
    not_predict,
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)

In [ ]:
# === EXP 4/4 — NoT × test_adjusted ===
m = run_exp(
    'configs/not_test_adjusted.yaml',
    not_predict,
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)

### Hybrid — PoT ↔ NoT per-category router

Each sample is dispatched to **PoT** or **NoT** based on its category, using the routing table in the YAML config:

| Route | Categories (arithmetic_saq) | Categories (test_adjusted) |
|-------|----------------------------|----------------------------|
| **PoT** | Hour Adj (24h/12h), Year/Month Shift, Time Comp, Week ID | add_subtract, trick, timezone |
| **NoT** | Date Computation, TZ Conversion, Application | compare, schedule, duration, multi_op |

Predictions record `routed_to` per sample. `run_hybrid_exp()` reports **per-route accuracy** (PoT branch vs. NoT branch) alongside the overall score, letting you directly measure how much each branch contributes.


In [ ]:
# === EXP 5/6 — Hybrid × arithmetic_saq ===
# Deterministic categories (Hour Adj, Month/Year Shift, Time Comp, Week ID) → PoT.
# Knowledge-intensive categories (Date Comp, TZ Conversion, Application) → NoT.
m = run_hybrid_exp(
    'configs/hybrid_arithmetic_saq.yaml',
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)


In [ ]:
# === EXP 6/6 — Hybrid × test_adjusted ===
# Pure-arithmetic types (add_subtract, trick, timezone) → PoT.
# Multi-entity relational types (compare, schedule, duration, multi_op) → NoT.
m = run_hybrid_exp(
    'configs/hybrid_test_adjusted.yaml',
    verbose=True,
    verbose_first_n=5,
    verbose_every=10,
    running_score_every=25,
    output_dir=DRIVE_OUT,
)
print(m)


## Debug helpers

Use these cells to audit prediction quality or probe individual samples.

In [ ]:
# === DEBUG A — audit predictions: show wrong and parse-fail samples ===
import json
from pathlib import Path

METHOD  = 'pot'            # 'pot' or 'not'
DATASET = 'arithmetic_saq' # 'arithmetic_saq' or 'test_adjusted'
N_SHOW  = 10

path = Path(DRIVE_OUT) / METHOD / DATASET / 'predictions.jsonl'
wrong, parse_fail_list = [], []
with open(path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r['predicted'] is None:
            parse_fail_list.append(r)
        elif not r['correct']:
            wrong.append(r)

print(f'parse_fail={len(parse_fail_list)}  wrong={len(wrong)}')

print('\n--- Parse failures (predicted=None) ---')
for r in parse_fail_list[:N_SHOW]:
    print(f"  [{r['category']}] gold={r['gold_norm']!r}")
    print(f"  Q: {r['question'][:100]}")

print('\n--- Wrong (but parsed) ---')
for r in wrong[:N_SHOW]:
    extra = f" (params={r.get('params')})" if r.get('params') else ''
    print(f"  [{r['category']}] gold={r['gold_norm']!r} pred={r['predicted']!r}{extra}")
    print(f"  Q: {r['question'][:100]}")

In [ ]:
# === DEBUG B — probe a single sample without running the full pipeline ===
import yaml

DATASET_NAME = 'arithmetic_saq'   # 'arithmetic_saq' or 'test_adjusted'
IDX          = 0                   # sample index
METHOD_NAME  = 'pot'               # 'pot' or 'not'

if DATASET_NAME == 'arithmetic_saq':
    samples = load_arithmetic_saq(SAQ_PATH, max_samples=IDX + 20, seed=42)
else:
    samples = load_test_adjusted(TEST_ADJ_PATH, max_samples=IDX + 20, seed=42)
s = samples[IDX]

print('=== Sample ===')
print(f'  Category : {s["category"]}')
print(f'  Question : {s["question"]}')
print(f'  Gold raw : {s["gold"]}')
print(f'  Gold norm: {_parse_gold(s["gold"])}')
print()

if METHOD_NAME == 'pot':
    result = pot_predict(MODEL, s)
    print('=== PoT Result ===')
    print(f'  Extracted params: {result.get("params")}')
    print(f'  Predicted       : {result["predicted"]!r}')
    print(f'  Fallback used   : {result.get("fallback_used")}')
else:
    result = not_predict(MODEL, s)
    print('=== NoT Result ===')
    print('  Raw output:')
    print(result.get('raw_output', ''))
    print(f'  Predicted: {result["predicted"]!r}')

correct = _is_correct(result['predicted'], s['gold'])
print(f'\n  Correct: {correct}')

In [ ]:
# === DEBUG C — summary table of all experiments ===
import pandas as pd, os

summary_csv = os.path.join(DRIVE_OUT, 'summary.csv')
df = pd.read_csv(summary_csv)
df = df.sort_values(['method', 'dataset']).reset_index(drop=True)
print(df.to_string(index=False))

In [ ]:
# === DEBUG D — PoT extraction trace: inspect params + executor output per category ===
# Runs on a small held-out slice to check which categories the executor handles vs. falls back.

DATASET_NAME = 'arithmetic_saq'  # or 'test_adjusted'
N_PER_CAT    = 2

if DATASET_NAME == 'arithmetic_saq':
    all_samples = load_arithmetic_saq(SAQ_PATH, max_samples=200, seed=0)
else:
    all_samples = load_test_adjusted(TEST_ADJ_PATH, max_samples=200, seed=0)

from collections import defaultdict
by_cat = defaultdict(list)
for s in all_samples:
    by_cat[s['category']].append(s)

for cat, slist in sorted(by_cat.items()):
    print(f'\n{'='*60}')
    print(f'Category: {cat}')
    for s in slist[:N_PER_CAT]:
        params = _pot_extract(MODEL, s['question'], s['category'])
        executed = _execute_params(params) if params else None
        fallback = executed is None
        print(f'  Q: {s["question"][:80]}')
        print(f'  params   : {params}')
        print(f'  executed : {executed!r}   (fallback={fallback})')
        print(f'  gold     : {s["gold"]!r}')

In [ ]:
# === DEBUG E — per-route accuracy breakdown within a Hybrid run ===
# Shows which categories each branch (PoT / NoT) handled and their accuracy.
import json, pandas as pd
from pathlib import Path
from collections import defaultdict

DATASET = 'arithmetic_saq'   # or 'test_adjusted'

path = Path(DRIVE_OUT) / 'hybrid' / DATASET / 'predictions.jsonl'
rows = [json.loads(l) for l in open(path, encoding='utf-8')]

# Per-route + per-category stats
route_data = defaultdict(lambda: {'correct': 0, 'total': 0,
                                   'cats': defaultdict(lambda: [0, 0])})
for r in rows:
    rt  = r.get('routed_to', '?')
    cat = r.get('category', '?')
    route_data[rt]['total'] += 1
    route_data[rt]['cats'][cat][1] += 1
    if r['correct']:
        route_data[rt]['correct'] += 1
        route_data[rt]['cats'][cat][0] += 1

print(f"Dataset: {DATASET}  ({len(rows)} samples)")
for route, d in sorted(route_data.items()):
    acc = d['correct'] / d['total'] if d['total'] else 0
    print(f"\n  Branch [{route.upper()}]  acc={acc:.3f}  ({d['correct']}/{d['total']}):")
    for cat, (c, t) in sorted(d['cats'].items()):
        print(f"    {cat:<32s}: {c/t:.3f} ({c}/{t})")

# Pivot: category × route — compare what PoT and NoT each achieve on their assigned tasks
table = [{'category': r['category'], 'route': r['routed_to'],
          'correct': int(r['correct'])} for r in rows]
df = pd.DataFrame(table)
pivot = df.groupby(['category', 'route'])['correct'].agg(['mean', 'count']).round(3)
print("\nPivot (category × route):")
print(pivot.to_string())


---
## MCQ Evaluation — model selects A / B / C / D from provided options

Uses **`arithmetic_mcq.csv`** as the test set and **`arithmetic_shots_mcq.csv`** as few-shot examples.  
Each sample has four labelled options; the model outputs a single letter.

| Step | Description |
|------|-------------|
| **MCQ SETUP 1** | Load test + shot CSVs, group shots by category |
| **MCQ SETUP 2** | Build few-shot prompt, extract A/B/C/D from raw output |
| **MCQ EXP**     | Run full evaluation, save `predictions.jsonl` + `metrics.json` |
| **MCQ DEBUG**   | Audit wrong predictions per category |

Run cells sequentially once (they share `MODEL` from SETUP 4 above).

In [ ]:
# === MCQ SETUP 1 — load test set + shot pool via src.prompts ===
import csv, json, time
from pathlib import Path
from collections import defaultdict

import pandas as pd

from src.data.schema import McqSample
from src.prompts.mcq_shot_pools import load_mcq_shots, get_mcq_shots, available_categories

MCQ_DIR        = 'F:/arithmetic'
MCQ_PATH       = f'{MCQ_DIR}/arithmetic_mcq.csv'
MCQ_SHOTS_PATH = f'{MCQ_DIR}/arithmetic_shots_mcq.csv'


def load_mcq_test(path, max_samples=None, seed=42):
    df = pd.read_csv(path, engine='python', on_bad_lines='warn')
    if max_samples and max_samples < len(df):
        df = df.sample(n=max_samples, random_state=seed).reset_index(drop=True)
    return [
        McqSample(
            sample_id=f'test-{i}',
            category=str(row['Category']),
            dataset='arithmetic_mcq',
            question=str(row['Question']),
            option_a=str(row['Option A']),
            option_b=str(row['Option B']),
            option_c=str(row['Option C']),
            option_d=str(row['Option D']),
            gold=str(row['Answer']).strip().upper(),
        )
        for i, (_, row) in enumerate(df.iterrows())
    ]


# Load
shot_pool    = load_mcq_shots(MCQ_SHOTS_PATH)
test_samples = load_mcq_test(MCQ_PATH)

print(f'Test samples : {len(test_samples)}')
print(f'Shot pool    : {sum(len(v) for v in shot_pool.values())} shots '
      f'across {len(shot_pool)} categories')
print(f'Categories   : {available_categories(shot_pool)}')


In [ ]:
# === MCQ SETUP 2 — Executor-first Compute-then-Match predict function ===
#
# Pipeline per sample:
#   1. parse_params(question, category)  — regex, no LLM, instant
#   2. execute(params)                   — deterministic Python, always correct
#      └─ if None → LLM compute fallback (build_compute_messages)
#   3. build_match_messages(computed)    — LLM matches text → A/B/C/D
#
# Separates computation (exact) from option selection (easy string match).
# LLM is only used for: (a) categories regex can't parse, (b) match step.

import re

from src.prompts.mcq_executor    import parse_params, execute
from src.prompts.mcq_templates   import (
    build_compute_messages, build_match_messages,
    extract_computed_answer, extract_letter,
)
from src.prompts.mcq_shot_pools  import get_mcq_shots


def mcq_predict(model, sample, pool, n_shots=3):
    """Executor-first Compute-then-Match for one McqSample.

    Returns dict: question, category, gold, predicted, correct,
        computed, source ('executor'|'llm'), raw_compute, raw_match.
    """
    question = sample['question']
    category = sample['category']

    # ── Stage 1a: try deterministic executor first ────────────────────────
    params   = parse_params(question, category)
    computed = execute(params) if params else None
    source   = 'executor'
    raw_compute = None

    # ── Stage 1b: LLM compute fallback if executor returned None ─────────
    if computed is None:
        source = 'llm'
        shots  = get_mcq_shots(pool, category=category, k=n_shots,
                               exclude_question=question)
        compute_msgs = build_compute_messages(sample, shots=shots)
        raw_compute  = model.generate(
            compute_msgs,
            max_new_tokens=150,
            temperature=0.0,
            do_sample=False,
            enable_thinking=False,
        )
        computed = extract_computed_answer(raw_compute)

    # ── Stage 2: LLM matches computed answer to A/B/C/D ──────────────────
    match_msgs = build_match_messages(computed, sample)
    raw_match  = model.generate(
        match_msgs,
        max_new_tokens=10,
        temperature=0.0,
        do_sample=False,
        enable_thinking=False,
    )
    predicted = extract_letter(raw_match)

    return {
        'question':    question,
        'category':    category,
        'gold':        sample['gold'],
        'predicted':   predicted,
        'correct':     predicted == sample['gold'],
        'computed':    computed,
        'source':      source,   # 'executor' or 'llm'
        'raw_compute': raw_compute,
        'raw_match':   raw_match,
    }


print('MCQ predict ready  —  executor-first Compute-then-Match.')
print('Executor categories:', list(__import__(
    "src.prompts.mcq_executor", fromlist=["_PARSERS"])._PARSERS.keys()))


In [ ]:
# === MCQ EXP — Compute-then-Match evaluation on arithmetic_mcq.csv ===
# N_SHOTS=3  : category-matched compute examples shown in Stage 1
# MAX_SAMPLES: integer for quick smoke-test; None = full dataset

N_SHOTS       = 3
MAX_SAMPLES   = None
VERBOSE_EVERY = 50
RUNNING_EVERY = 200

samples = load_mcq_test(MCQ_PATH, max_samples=MAX_SAMPLES)
n = len(samples)

out_dir = Path(DRIVE_OUT) / 'mcq' / 'arithmetic_mcq'
out_dir.mkdir(parents=True, exist_ok=True)
pred_file = out_dir / 'predictions.jsonl'

print(f'[MCQ EXP]  strategy=compute_then_match  n={n}  n_shots={N_SHOTS}')
print(f'           output={pred_file}')

correct_n = parse_fail = 0
cat_stats = defaultdict(lambda: [0, 0])
records   = []

with open(pred_file, 'w', encoding='utf-8') as pf:
    for i, s in enumerate(samples):
        t0  = time.time()
        res = mcq_predict(MODEL, s, shot_pool, n_shots=N_SHOTS)
        elapsed = time.time() - t0

        pred = res['predicted']
        if not pred:
            parse_fail += 1
        if res['correct']:
            correct_n += 1

        cat = s['category']
        cat_stats[cat][1] += 1
        if res['correct']:
            cat_stats[cat][0] += 1

        rec = {
            'idx':         i,
            'question':    s['question'],
            'category':    cat,
            'options':     {'A': s['option_a'], 'B': s['option_b'],
                            'C': s['option_c'], 'D': s['option_d']},
            'gold':        res['gold'],
            'predicted':   pred,
            'correct':     res['correct'],
            'computed':    res['computed'],
            'elapsed_sec': round(elapsed, 3),
        }
        records.append(rec)
        pf.write(json.dumps(rec, ensure_ascii=False) + '\n')
        pf.flush()

        if i < 5 or (VERBOSE_EVERY > 0 and (i + 1) % VERBOSE_EVERY == 0):
            mark = '\u2713' if res['correct'] else '\u2717'

            print(f'\n[{i+1:>6}/{n}] {mark} [{cat}]')
            print(f'Question : {s["question"]}')
            print(f'A. {s["option_a"]}')
            print(f'B. {s["option_b"]}')
            print(f'C. {s["option_c"]}')
            print(f'D. {s["option_d"]}')
            print(f'Computed : {res["computed"]!r}')
            print(f'Gold     : {res["gold"]}')
            print(f'Pred     : {pred}')
            print(f'Time     : {elapsed:.2f}s')

        if RUNNING_EVERY > 0 and (i + 1) % RUNNING_EVERY == 0:
            acc_r = correct_n / (i + 1)
            print(f'  [{i+1:>6}/{n}] running acc={acc_r:.3f} '
                  f'({correct_n}/{i+1})  parse_fail={parse_fail}')

# Final metrics
acc = correct_n / n if n else 0.0
cat_acc = {k: round(v[0] / v[1], 3) for k, v in sorted(cat_stats.items()) if v[1]}

metrics = {
    'accuracy':     round(acc, 4),
    'correct':      correct_n,
    'total':        n,
    'parse_fail':   parse_fail,
    'per_category': cat_acc,
}
(out_dir / 'metrics.json').write_text(
    json.dumps({'method': 'compute_then_match', 'n_shots': N_SHOTS,
                'dataset': 'arithmetic_mcq', 'metrics': metrics},
               indent=2, ensure_ascii=False),
    encoding='utf-8',
)

summary_path = Path(DRIVE_OUT) / 'summary.csv'
header_needed = not summary_path.exists()
row = {
    'experiment': f'ctm_{N_SHOTS}shot', 'method': 'compute_then_match',
    'dataset': 'arithmetic_mcq', 'accuracy': metrics['accuracy'],
    'correct': correct_n, 'total': n, 'parse_fail': parse_fail,
}
with open(summary_path, 'a', encoding='utf-8', newline='') as sf:
    import csv as _csv
    w = _csv.DictWriter(sf, fieldnames=list(row.keys()))
    if header_needed:
        w.writeheader()
    w.writerow(row)

print(f'\n── MCQ Results (Compute-then-Match) ────────────────────────')
print(f'  n_shots      : {N_SHOTS}')
print(f'  accuracy     : {acc:.4f}  ({correct_n}/{n})')
print(f'  parse_fail   : {parse_fail}')
for cat, a in cat_acc.items():
    c, t = cat_stats[cat]
    print(f'  {cat:<32s}: {a:.3f}  ({c}/{t})')
print(f'  predictions  : {pred_file}')
print(f'────────────────────────────────────────────────────────────')
print(metrics)


In [ ]:
# === MCQ DEBUG — audit wrong and parse-fail predictions ===
import json
from pathlib import Path

N_SHOW  = 10
path = Path(DRIVE_OUT) / 'mcq' / 'arithmetic_mcq' / 'predictions.jsonl'

wrong, parse_fail_list = [], []
with open(path, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if not r['predicted']:
            parse_fail_list.append(r)
        elif not r['correct']:
            wrong.append(r)

print(f'parse_fail={len(parse_fail_list)}  wrong={len(wrong)}')

print('\n--- Parse failures (predicted=None or empty) ---')
for r in parse_fail_list[:N_SHOW]:
    print(f"  [{r['category']}]  gold={r['gold']!r}  raw={r.get('raw_output','')[:40]!r}")
    print(f"  Q: {r['question'][:100]}")

print('\n--- Wrong answers (by category) ---')
# Group wrong by category so you can spot patterns
from collections import defaultdict
wrong_by_cat = defaultdict(list)
for r in wrong:
    wrong_by_cat[r['category']].append(r)

for cat in sorted(wrong_by_cat.keys()):
    items = wrong_by_cat[cat]
    opts_ex = items[0].get('options', {})
    print(f'\n  [{cat}]  wrong={len(items)}')
    for r in items[:N_SHOW]:
        opts = r.get('options', {})
        gold_text = opts.get(r['gold'], '?')
        pred_text = opts.get(r['predicted'], '?') if r['predicted'] else 'None'
        print(f"    gold={r['gold']}({gold_text!r})  pred={r['predicted']}({pred_text!r})")
        print(f"    Q: {r['question'][:90]}")

In [ ]:
# === MCQ BASELINE SETUP — Direct MCQ predict function ===
# Baseline strategy: model sees Q + 4 options in a single call → outputs A/B/C/D.
# No intermediate computation step; model must reason and select simultaneously.
# Compared against Compute-then-Match to measure the value of the two-stage pipeline.

from src.prompts.mcq_templates  import build_mcq_messages, extract_letter
from src.prompts.mcq_shot_pools import get_mcq_shots


def direct_mcq_predict(model, sample, pool, n_shots=0):
    """Direct MCQ baseline: single inference → A/B/C/D.

    Args:
        n_shots: 0 = zero-shot, >0 = category-matched few-shot examples.
    """
    shots = []
    if n_shots > 0:
        shots = get_mcq_shots(pool, category=sample['category'],
                              k=n_shots, exclude_question=sample['question'])

    msgs = build_mcq_messages(sample, shots=shots)
    raw, usage = model.generate_with_usage(
        msgs,
        max_new_tokens=10,
        temperature=0.0,
        do_sample=False,
        enable_thinking=False,
    )
    predicted = extract_letter(raw)

    return {
        'question':  sample['question'],
        'category':  sample['category'],
        'gold':      sample['gold'],
        'predicted': predicted,
        'correct':   predicted == sample['gold'],
        'raw':       raw,
        'usage':     usage,
    }


print('Direct MCQ baseline predict ready.')
print('  n_shots=0 → zero-shot | n_shots=3 → few-shot')


In [ ]:
# === MCQ EXP — Zero-shot Direct MCQ baseline ===
# Strategy : direct A/B/C/D, no examples, no intermediate compute step.
# MAX_SAMPLES: integer for quick smoke-test; None = full dataset

N_SHOTS       = 0
MAX_SAMPLES   = None
VERBOSE_EVERY = 50
RUNNING_EVERY = 200

samples = load_mcq_test(MCQ_PATH, max_samples=MAX_SAMPLES)
n = len(samples)

out_dir = Path(DRIVE_OUT) / 'mcq' / 'arithmetic_mcq'
out_dir.mkdir(parents=True, exist_ok=True)
pred_file = out_dir / 'predictions_zeroshot.jsonl'

print(f'[MCQ EXP]  strategy=direct_zeroshot  n={n}')
print(f'           output={pred_file}')

correct_n = parse_fail = 0
tok_prompt = tok_comp  = 0
cat_stats  = defaultdict(lambda: [0, 0])
records    = []

with open(pred_file, 'w', encoding='utf-8') as pf:
    for i, s in enumerate(samples):
        t0  = time.time()
        res = direct_mcq_predict(MODEL, s, shot_pool, n_shots=N_SHOTS)
        elapsed = time.time() - t0

        pred = res['predicted']
        if not pred:
            parse_fail += 1
        if res['correct']:
            correct_n += 1

        cat = s['category']
        cat_stats[cat][1] += 1
        if res['correct']:
            cat_stats[cat][0] += 1

        tok_prompt += res['usage']['prompt_tokens']
        tok_comp   += res['usage']['completion_tokens']

        rec = {
            'idx':         i,
            'question':    s['question'],
            'category':    cat,
            'options':     {'A': s['option_a'], 'B': s['option_b'],
                            'C': s['option_c'], 'D': s['option_d']},
            'gold':        res['gold'],
            'predicted':   pred,
            'correct':     res['correct'],
            'elapsed_sec': round(elapsed, 3),
        }
        records.append(rec)
        pf.write(json.dumps(rec, ensure_ascii=False) + '\n')
        pf.flush()

        if i < 5 or (VERBOSE_EVERY > 0 and (i + 1) % VERBOSE_EVERY == 0):
            mark = '\u2713' if res['correct'] else '\u2717'
            print(f'\n[{i+1:>6}/{n}] {mark} [{cat}]')
            print(f'Gold: {res["gold"]}  Pred: {pred}  Raw: {res["raw"]!r}')
            print(f'Q: {s["question"][:80]}')

        if RUNNING_EVERY > 0 and (i + 1) % RUNNING_EVERY == 0:
            print(f'  [{i+1:>6}/{n}] running acc={correct_n/(i+1):.3f} '
                  f'({correct_n}/{i+1})  parse_fail={parse_fail}')

# Final metrics
acc     = correct_n / n if n else 0.0
cat_acc = {k: round(v[0] / v[1], 3) for k, v in sorted(cat_stats.items()) if v[1]}
metrics = {
    'accuracy':     round(acc, 4),
    'correct':      correct_n,
    'total':        n,
    'parse_fail':   parse_fail,
    'per_category': cat_acc,
    'tokens':       {'prompt': tok_prompt, 'completion': tok_comp,
                     'total': tok_prompt + tok_comp,
                     'avg_per_sample': round((tok_prompt + tok_comp) / n, 1)},
}
(out_dir / 'metrics_zeroshot.json').write_text(
    json.dumps({'method': 'direct_zeroshot', 'n_shots': 0,
                'dataset': 'arithmetic_mcq', 'metrics': metrics},
               indent=2, ensure_ascii=False),
    encoding='utf-8',
)

summary_path = Path(DRIVE_OUT) / 'summary.csv'
header_needed = not summary_path.exists()
row = {
    'experiment': 'direct_0shot', 'method': 'direct_zeroshot',
    'dataset': 'arithmetic_mcq', 'accuracy': metrics['accuracy'],
    'correct': correct_n, 'total': n, 'parse_fail': parse_fail,
}
with open(summary_path, 'a', encoding='utf-8', newline='') as sf:
    import csv as _csv
    w = _csv.DictWriter(sf, fieldnames=list(row.keys()))
    if header_needed:
        w.writeheader()
    w.writerow(row)

print(f'\n── MCQ Results (Zero-shot Direct) ──────────────────────────')
print(f'  accuracy     : {acc:.4f}  ({correct_n}/{n})')
print(f'  parse_fail   : {parse_fail}')
print(f'  tokens total : {tok_prompt + tok_comp}  avg/sample: {(tok_prompt+tok_comp)/n:.1f}')
for cat, a in cat_acc.items():
    c, t = cat_stats[cat]
    print(f'  {cat:<32s}: {a:.3f}  ({c}/{t})')
print(f'  predictions  : {pred_file}')
print(f'────────────────────────────────────────────────────────────')


In [ ]:
# === MCQ EXP — Few-shot Direct MCQ baseline ===
# Strategy : direct A/B/C/D with k category-matched examples before each question.
# MAX_SAMPLES: integer for quick smoke-test; None = full dataset

N_SHOTS       = 3
MAX_SAMPLES   = None
VERBOSE_EVERY = 50
RUNNING_EVERY = 200

samples = load_mcq_test(MCQ_PATH, max_samples=MAX_SAMPLES)
n = len(samples)

out_dir = Path(DRIVE_OUT) / 'mcq' / 'arithmetic_mcq'
out_dir.mkdir(parents=True, exist_ok=True)
pred_file = out_dir / f'predictions_fewshot{N_SHOTS}.jsonl'

print(f'[MCQ EXP]  strategy=direct_fewshot  n_shots={N_SHOTS}  n={n}')
print(f'           output={pred_file}')

correct_n = parse_fail = 0
tok_prompt = tok_comp  = 0
cat_stats  = defaultdict(lambda: [0, 0])
records    = []

with open(pred_file, 'w', encoding='utf-8') as pf:
    for i, s in enumerate(samples):
        t0  = time.time()
        res = direct_mcq_predict(MODEL, s, shot_pool, n_shots=N_SHOTS)
        elapsed = time.time() - t0

        pred = res['predicted']
        if not pred:
            parse_fail += 1
        if res['correct']:
            correct_n += 1

        cat = s['category']
        cat_stats[cat][1] += 1
        if res['correct']:
            cat_stats[cat][0] += 1

        tok_prompt += res['usage']['prompt_tokens']
        tok_comp   += res['usage']['completion_tokens']

        rec = {
            'idx':         i,
            'question':    s['question'],
            'category':    cat,
            'options':     {'A': s['option_a'], 'B': s['option_b'],
                            'C': s['option_c'], 'D': s['option_d']},
            'gold':        res['gold'],
            'predicted':   pred,
            'correct':     res['correct'],
            'elapsed_sec': round(elapsed, 3),
        }
        records.append(rec)
        pf.write(json.dumps(rec, ensure_ascii=False) + '\n')
        pf.flush()

        if i < 5 or (VERBOSE_EVERY > 0 and (i + 1) % VERBOSE_EVERY == 0):
            mark = '\u2713' if res['correct'] else '\u2717'
            print(f'\n[{i+1:>6}/{n}] {mark} [{cat}]')
            print(f'Gold: {res["gold"]}  Pred: {pred}  Raw: {res["raw"]!r}')
            print(f'Q: {s["question"][:80]}')

        if RUNNING_EVERY > 0 and (i + 1) % RUNNING_EVERY == 0:
            print(f'  [{i+1:>6}/{n}] running acc={correct_n/(i+1):.3f} '
                  f'({correct_n}/{i+1})  parse_fail={parse_fail}')

# Final metrics
acc     = correct_n / n if n else 0.0
cat_acc = {k: round(v[0] / v[1], 3) for k, v in sorted(cat_stats.items()) if v[1]}
metrics = {
    'accuracy':     round(acc, 4),
    'correct':      correct_n,
    'total':        n,
    'parse_fail':   parse_fail,
    'per_category': cat_acc,
    'tokens':       {'prompt': tok_prompt, 'completion': tok_comp,
                     'total': tok_prompt + tok_comp,
                     'avg_per_sample': round((tok_prompt + tok_comp) / n, 1)},
}
(out_dir / f'metrics_fewshot{N_SHOTS}.json').write_text(
    json.dumps({'method': 'direct_fewshot', 'n_shots': N_SHOTS,
                'dataset': 'arithmetic_mcq', 'metrics': metrics},
               indent=2, ensure_ascii=False),
    encoding='utf-8',
)

summary_path = Path(DRIVE_OUT) / 'summary.csv'
header_needed = not summary_path.exists()
row = {
    'experiment': f'direct_{N_SHOTS}shot', 'method': 'direct_fewshot',
    'dataset': 'arithmetic_mcq', 'accuracy': metrics['accuracy'],
    'correct': correct_n, 'total': n, 'parse_fail': parse_fail,
}
with open(summary_path, 'a', encoding='utf-8', newline='') as sf:
    import csv as _csv
    w = _csv.DictWriter(sf, fieldnames=list(row.keys()))
    if header_needed:
        w.writeheader()
    w.writerow(row)

print(f'\n── MCQ Results (Few-shot Direct, n_shots={N_SHOTS}) ─────────')
print(f'  accuracy     : {acc:.4f}  ({correct_n}/{n})')
print(f'  parse_fail   : {parse_fail}')
print(f'  tokens total : {tok_prompt + tok_comp}  avg/sample: {(tok_prompt+tok_comp)/n:.1f}')
for cat, a in cat_acc.items():
    c, t = cat_stats[cat]
    print(f'  {cat:<32s}: {a:.3f}  ({c}/{t})')
print(f'  predictions  : {pred_file}')
print(f'────────────────────────────────────────────────────────────')


---
## Real-World Evaluator

Paste any temporal arithmetic question (with or without options).
The system auto-detects the category, runs the executor or LLM, and returns the answer.

**Input format** (paste as a block):
```
Question: <question text>
A. <option>
B. <option>
C. <option>
D. <option>
```
Options are optional — if omitted, only the computed answer is shown (no letter selection).


In [ ]:
# === REAL-WORLD EVALUATOR ===
# Paste a question block below. Options are optional.
# The system will auto-detect the category and compute the answer.

import re, textwrap
from src.prompts.mcq_executor  import detect_category, parse_params, execute
from src.prompts.mcq_templates import (
    build_compute_messages, build_match_messages,
    extract_computed_answer, extract_letter,
)
from src.prompts.mcq_shot_pools import get_mcq_shots
from src.data.schema import McqSample


def parse_question_block(block: str):
    lines = [l.strip() for l in block.strip().splitlines() if l.strip()]
    question_lines = []
    options = {}
    opt_re = re.compile(r'^([A-D])[.):][ \t]*(.*)', re.IGNORECASE)
    for line in lines:
        if re.match(r'^question\s*:', line, re.IGNORECASE):
            line = re.sub(r'^question\s*:\s*', '', line, flags=re.IGNORECASE)
        m = opt_re.match(line)
        if m:
            options[m.group(1).upper()] = m.group(2).strip()
        else:
            question_lines.append(line)
    question = ' '.join(question_lines).strip()
    return question, options


def evaluate(block: str, n_shots: int = 3, gold: str = None):
    question, options = parse_question_block(block)
    if not question:
        print('[ERROR] No question found in input.')
        return

    category = detect_category(question)
    print(f'  Category  : {category or "unknown (will use LLM compute)"}')

    params   = parse_params(question, category) if category else None
    computed = execute(params) if params else None
    source   = 'executor'

    if computed is None:
        source = 'llm-compute'
        dummy  = McqSample(
            sample_id='live', category=category or 'unknown',
            dataset='live', question=question,
            option_a='', option_b='', option_c='', option_d='', gold='',
        )
        shots = get_mcq_shots(shot_pool, category=category or '', k=n_shots,
                              exclude_question=question) if category else []
        compute_msgs = build_compute_messages(dummy, shots=shots)
        raw_compute, usage_c = MODEL.generate_with_usage(
            compute_msgs, max_new_tokens=400, temperature=0.0,
            do_sample=False, enable_thinking=False,
        )
        computed = extract_computed_answer(raw_compute)
        print(f'  LLM raw   : {raw_compute.strip()[:120]}')
        print(f'  Tok(comp) : prompt={usage_c["prompt_tokens"]}  completion={usage_c["completion_tokens"]}')
    else:
        usage_c = {'prompt_tokens': 0, 'completion_tokens': 0}

    print(f'  Computed  : {computed!r}  [{source}]')

    if computed is None:
        print('[ERROR] Could not extract a final answer — try increasing max_new_tokens.')
        return

    predicted = None
    usage_m   = {'prompt_tokens': 0, 'completion_tokens': 0}

    if options:
        dummy = McqSample(
            sample_id='live', category=category or 'unknown',
            dataset='live', question=question,
            option_a=options.get('A', ''),
            option_b=options.get('B', ''),
            option_c=options.get('C', ''),
            option_d=options.get('D', ''),
            gold='',
        )
        match_msgs = build_match_messages(computed, dummy)
        raw_match, usage_m = MODEL.generate_with_usage(
            match_msgs, max_new_tokens=10, temperature=0.0,
            do_sample=False, enable_thinking=False,
        )
        predicted = extract_letter(raw_match)

    total_tok = (usage_c['prompt_tokens'] + usage_c['completion_tokens'] +
                 usage_m['prompt_tokens'] + usage_m['completion_tokens'])

    print()
    print('  +' + '-'*51 + '+')
    print(f'  |  Computed answer : {computed:<29s}|')
    if options:
        pred_text = options.get(predicted, '?') if predicted else '?'
        print(f'  |  Predicted letter: {(predicted or "?"):<29s}|')
        print(f'  |  Option text     : {pred_text:<29s}|')
        if gold:
            correct = predicted == gold.upper()
            mark = "CORRECT" if correct else f"WRONG (gold={gold.upper()})"
            print(f'  |  Result          : {mark:<29s}|')
    print(f'  |  Tokens used     : {total_tok:<29d}|')
    print('  +' + '-'*51 + '+')

    if options:
        print()
        for letter in 'ABCD':
            text = options.get(letter, '')
            marker = ' <-- predicted' if letter == predicted else (
                     ' <-- GOLD'     if letter == gold        else '')
            print(f'    {letter}) {text}{marker}')

    return {
        'question': question, 'category': category, 'computed': computed,
        'source': source, 'predicted': predicted, 'options': options, 'tokens': total_tok,
    }


QUESTION_BLOCK = """
What is 23:48 - 01:31?
A. 23:33
B. 21:09
C. 22:17
D. 20:08
"""

GOLD = 'C'   # set to None if unknown

print('=' * 55)
print('  Temporal Arithmetic Evaluator')
print('=' * 55)
print(f'  Question  : {QUESTION_BLOCK.strip().splitlines()[0][:50]}')
result = evaluate(QUESTION_BLOCK, n_shots=3, gold=GOLD)
